# OOD Detection via Posterior Variability

This notebook computes the posterior variability score $D_v$ to detect
whether an observation is out-of-distribution (OOD).

**Idea:** If N independently trained density estimators produce very different
posteriors for the same observation, the data is likely OOD.

$$D_v(\{q_i\}_{i=1}^{N_p}) = \frac{1}{N_p(N_p - 1)} \sum_{i \neq j} D_{KL}(q_i \| q_j)$$

In [ ]:
import sys
from pathlib import Path
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt

project_root = Path("../../..").resolve()
sys.path.insert(0, str(project_root))

from sbi4atmret.config.configs import BaseConfig
from sbi4atmret.models.ModelBase import BaseModel
from sbi4atmret.models.meta_learner import load_base_models
from sbi4atmret.evaluation.ood_tests import (
    compute_ood_score,
    plot_variability,
    pairwise_kl_divergence,
    posterior_variability_score,
)

## 1. Load Config and Base Models

In [ ]:
config_path = project_root / "experiments/config_MiriGeminiHST_cloudfree.yaml"

with open(config_path) as f:
    config_dict = yaml.safe_load(f)

config = BaseConfig(**config_dict)

# Parameter names for plotting
param_names = [p.name for p in config.prior.parameters]
n_params = len(param_names)
print(f"{n_params} parameters: {param_names[:5]}...")

In [ ]:
# Checkpoint paths for your independently trained models
checkpoint_paths = [
    Path("path/to/model_seed1/states_800.pth"),
    Path("path/to/model_seed2/states_800.pth"),
    Path("path/to/model_seed3/states_800.pth"),
]

device = "cuda" if torch.cuda.is_available() else "cpu"

base_models = load_base_models(
    checkpoint_paths,
    model_builder=lambda: BaseModel(config).build(),
    device=device,
)

print(f"Loaded {len(base_models)} frozen base models")

## 2. Load Observation

In [ ]:
# Load your observation
# Replace with your actual observation loading
# x_obs = torch.from_numpy(observation.full_observation).unsqueeze(0).float()

# Placeholder:
# x_obs = torch.randn(1, 1298 + 434)  # miri + gemini dims

print(f"Observation shape: {x_obs.shape}")

## 3. Compute OOD Score (Gaussian method)

In [ ]:
result = compute_ood_score(
    base_models,
    x_obs,
    n_samples=2048,
    method="gaussian",
    device=device,
)

print(f"Variability score D_v = {result.variability_score:.4f}")
print(f"KL matrix shape: {result.kl_matrix.shape}")
print(f"\nKL matrix:\n{np.round(result.kl_matrix, 3)}")

## 4. Visualize

In [ ]:
fig = plot_variability(result, param_names=param_names)
plt.show()

## 5. Compare In-Distribution vs OOD

Run the same analysis on a known in-distribution sample (from the test set)
to calibrate what "normal" D_v looks like.

In [ ]:
# Compute D_v for multiple test samples to get a reference distribution
# test_observations: (N_test, D_obs) tensor of in-distribution test observations

# dv_in_dist = []
# for i in range(min(50, len(test_observations))):
#     x_test = test_observations[i:i+1].float()
#     r = compute_ood_score(base_models, x_test, n_samples=1024, device=device)
#     dv_in_dist.append(r.variability_score)
#
# dv_in_dist = np.array(dv_in_dist)
# print(f"In-distribution D_v: mean={dv_in_dist.mean():.4f}, std={dv_in_dist.std():.4f}")
# print(f"Observation D_v: {result.variability_score:.4f}")
# print(f"Z-score: {(result.variability_score - dv_in_dist.mean()) / dv_in_dist.std():.2f}")

In [ ]:
# # Plot distribution of D_v for in-distribution vs the observation
# fig, ax = plt.subplots(figsize=(7, 4))
# ax.hist(dv_in_dist, bins=20, alpha=0.7, color="steelblue", label="In-distribution")
# ax.axvline(result.variability_score, color="red", linewidth=2,
#            linestyle="--", label=f"Observation ($D_v={result.variability_score:.3f}$)")
# ax.set_xlabel(r"$D_v$")
# ax.set_ylabel("Count")
# ax.set_title("Posterior Variability: In-Distribution vs Observation")
# ax.legend()
# plt.tight_layout()
# plt.show()

## 6. Monte Carlo KL (Non-parametric)

If the posteriors are highly non-Gaussian, use the k-NN based MC estimate.

In [ ]:
result_mc = compute_ood_score(
    base_models,
    x_obs,
    n_samples=2048,
    method="mc",
    device=device,
)

print(f"Gaussian D_v = {result.variability_score:.4f}")
print(f"MC (k-NN) D_v = {result_mc.variability_score:.4f}")

In [ ]:
fig_mc = plot_variability(result_mc, param_names=param_names)
plt.show()

## 7. Per-Parameter Analysis

Identify which parameters the models disagree on most.

In [ ]:
# Sort parameters by variability
sorted_idx = np.argsort(result.per_param_scores)[::-1]

print("Parameters ranked by disagreement:")
for rank, idx in enumerate(sorted_idx[:10]):
    print(f"  {rank+1}. {param_names[idx]:>15s}: D_KL = {result.per_param_scores[idx]:.4f}")